In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from scipy.optimize import curve_fit

In [ ]:
dataF = np.load('dataF.npy')
dataN = np.load('dataN.npy')
dataO = np.load('dataO.npy')
dataS = np.load('dataS.npy')
dataZ = np.load('dataZ.npy')

folders = [dataF, dataN, dataO, dataS, dataZ]
filenames = ['F', 'N', 'O', 'S', 'Z']

data = dataF[0]

scales = 9
time = np.linspace(0, 23.6, len(dataF[0]))
epochs = np.logspace(
    np.log10(4),
    np.log10(4096/4),
    scales)

x_extrapolate = np.linspace(4, 4096/4, 100)

print(epochs)

In [ ]:
def prepros(tseries):
    mean = np.mean(tseries)
    mean_centered = tseries - mean
    prepros = np.cumsum(mean_centered)
    return prepros

def linear(x, m, b):
    return m*x + b

def sigmoid1(x, L ,x0, k, b):
    return L / (1 + np.exp(-k*(x-x0))) + b

def gaussian(x, amplitude, mean, sigma):
    return amplitude * np.exp(-((x - mean) ** 2) / (2 * sigma ** 2))

def sigmoid(x, L, x0, k, b):
    z = k * (x - x0)
    z = np.clip(z, -60, 60)  # prevent overflow
    return L / (1 + np.exp(-z)) + b

def fitting_sig(x, y, p0=None):
    # default starting guess
    if p0 is None:
        L0 = np.max(y) - np.min(y)
        x0_0 = np.median(np.log(x))
        k0 = 1.0
        b0 = np.min(y)
        p0 = [L0, x0_0, k0, b0]

    # residuals
    def residuals(params, x_obs, y_obs):
        return sigmoid(x_obs, *params) - y_obs

    result = least_squares(
        residuals,
        p0,
        args=(x, y),
        method="trf",
        max_nfev=20000
    )

    return result.x


def fitting_lin(segment_y, segment_x):
    popt, pcov = curve_fit(linear, segment_x, segment_y)
    m, b = popt
    return m, b

def RMS_i(f1,f2,n):
    rms_i = np.sqrt( (1/n) * np.sum((f1 - f2)**2) )
    return rms_i

def RMS(rms_i_list):
    F_n = np.sqrt( (1/len(rms_i_list)) * np.sum(np.array(rms_i_list)**2) )
    return F_n

def DFA(prepros):
    RMS_lin_list = []
    RMS_sig_list = []

    for i in epochs:
        boundaries = np.arange(0,4097,i).astype(int)
        rms_i_list = []

        # Epoch processing
        for j in range(len(boundaries)-1):
            segment_x = [k for k in range(int(boundaries[j]), int(boundaries[j+1]))]
            segment_y = prepros[boundaries[j]:boundaries[j+1]]

            # Fitting
            m, b = fitting_lin(segment_y, segment_x)
            L, x0, k, b = fitting_sig(segment_y, segment_x)

            # RMS Computation 
            rms_lin_i = RMS_i(segment_y, linear(np.array(segment_x), m, b), i)
            rms_sig_i = RMS_i(segment_y, sigmoid(np.array(segment_x), L, x0, k, b), i)
            rms_lin_i_list.append(rms_lin_i)
            rms_sig_i_list.append(rms_sig_i)

        F_lin_n = RMS(rms_lin_i_list)
        F_sig_n = RMS(rms_sig_i_list)
        RMS_lin_list.append(F_lin_n)
        RMS_sig_list.append(F_sig_n)
    return RMS_lin_list, RMS_sig_list

In [ ]:
# DATA GENERATION
for folder in range(len(folders)):
    DFA_results = []
    DFA_Hurst = []
    DFA_linfit = []

    for file in folders[folder]:
        data = file
        processed = prepros(data)
        DFA_result = DFA(processed)
        DFA_results.append(DFA_result)

        m1 , b1 = fitting(np.log(DFA_result), np.log(epochs))
        DFA_linfit.append([m1, b1])
        DFA_Hurst.append(m1)    

    np.array(DFA_results).astype(float)
    np.array(DFA_Hurst)
    np.array(DFA_linfit).astype(float)
    np.save(f'DFA_{filenames[folder]}_results.npy', np.array(DFA_results))
    np.save(f'DFA_{filenames[folder]}_Hurst.npy', np.array(DFA_Hurst))
    np.save(f'DFA_{filenames[folder]}_sigfit.npy', np.array(DFA_linfit))


# Plotting
DFA datapoints = "DFA_X_results.npy" 
Hurst Exponents = "DFA_X_Hurst.npy"

x-axis for epochs = "epochs"